# 03 — Java Platform Threads and Virtual Threads

**Target:** Apache Spark 4.2.0 runtime; JDK 21+ for virtual threads; Python 3.10+.

Run cells in order. These labs use small local workloads and avoid deliberately exhausting cluster resources. Read each *Why* before changing the experiment.

## Question
How do Java platform threads and virtual threads behave on a blocking workload?

**Why this matters:** Loom can reduce the cost of representing many blocked Java tasks. It does not remove CPU limits or make retained objects free. This lab runs a separate Java program and does not change Spark internals.

In [ ]:
import subprocess, tempfile, pathlib, textwrap, shutil
print('java:', shutil.which('java'), 'javac:', shutil.which('javac'))
source = r'''
import java.util.concurrent.*;
public class LoomCompare {
  static long run(ExecutorService pool, int n) throws Exception {
    long start=System.nanoTime();
    for (int i=0;i<n;i++) pool.submit(() -> { try { Thread.sleep(100); } catch (InterruptedException e) { Thread.currentThread().interrupt(); } });
    pool.shutdown(); pool.awaitTermination(2, TimeUnit.MINUTES);
    return TimeUnit.NANOSECONDS.toMillis(System.nanoTime()-start);
  }
  public static void main(String[] a) throws Exception {
    int n=500;
    System.out.println("platform ms="+run(Executors.newFixedThreadPool(100),n));
    System.out.println("virtual ms="+run(Executors.newVirtualThreadPerTaskExecutor(),n));
  }
}
'''
work = pathlib.Path(tempfile.mkdtemp(prefix='loom-lab-'))
(work/'LoomCompare.java').write_text(source, encoding='utf-8')
compiled = subprocess.run(['javac', str(work/'LoomCompare.java')], capture_output=True, text=True)
print(compiled.stdout, compiled.stderr)
if compiled.returncode == 0:
    run = subprocess.run(['java','-cp',str(work),'LoomCompare'],capture_output=True,text=True,timeout=180)
    print(run.stdout, run.stderr)
    print('exit:',run.returncode)

## Interpret carefully
The fixed-size platform pool is intentionally bounded; the virtual executor creates one virtual thread per task. This is a tiny blocking demonstration, not a performance benchmark. Run several times and record JDK version.

**Expected:** virtual threads can increase concurrency for blocking tasks. Results depend on the machine and scheduler. Add a CPU-bound variant to see that thread type does not add CPU cores.

**Why:** JEP 444 stores suspended virtual-thread stacks in heap chunks and schedules them over carrier platform threads. Heap retention, thread locals and downstream resource limits still matter. JDK 21+ required. Spark pools are not converted automatically.